**Import packages and dependecies**

In [1]:
%pip install -q -e ..

Note: you may need to restart the kernel to use updated packages.


ERROR: file:///C:/Users/tcphan/OneDrive/Documents/Data%20Science%20Projects/topological does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import warnings

warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs
import kagglehub

from tda.dimensionality_reduction import UMAP
from tda.matching import mapper_bin_and_cluster

### 1. Load in data inputs

The dataset below contains roughly 100,000 member records on various health conditions and risk factors, including:

- Demographics (e.g. age, gender)
- Chronic conditions (e.g. diabetes, hypertension)
- Biometric (e.g. blood pressure, LDL)
- Medical risk factors (e.g. smoking, alcohol usage)
- Prior claims utilization (e.g. # hospitalizations in last 3 years)

For further information about the data, see [Medical Insurance Cost Prediction](https://www.kaggle.com/datasets/mohankrishnathalla/medical-insurance-cost-prediction).

In [3]:
# Download data fram Kaggle
path = kagglehub.dataset_download(
    "mohankrishnathalla/medical-insurance-cost-prediction"
)


health_risk_factors_df = pl.read_csv(f"{path}/medical_insurance.csv")

# Show data
health_risk_factors_df.limit(50).show()

person_id,age,sex,region,urban_rural,income,education,marital_status,employment_status,household_size,dependents,bmi,smoker,alcohol_freq,visits_last_year,hospitalizations_last_3yrs,days_hospitalized_last_3yrs,medication_count,systolic_bp,diastolic_bp,ldl,hba1c,plan_type,network_tier,deductible,copay,policy_term_years,policy_changes_last_2yrs,provider_quality,risk_score,annual_medical_cost,annual_premium,monthly_premium,claims_count,avg_claim_amount,total_claims_paid,chronic_count,hypertension,diabetes,asthma,copd,cardiovascular_disease,cancer_history,kidney_disease,liver_disease,arthritis,mental_health,proc_imaging_count,proc_surgery_count,proc_physio_count,proc_consult_count,proc_lab_count,is_high_risk,had_major_procedure
i64,i64,str,str,str,f64,str,str,str,i64,i64,f64,str,str,i64,i64,i64,i64,f64,f64,f64,f64,str,str,i64,i64,i64,i64,f64,f64,f64,f64,f64,i64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
75722,52,"""Female""","""North""","""Suburban""",22700.0,"""Doctorate""","""Married""","""Retired""",3,1,27.4,"""Never""","""None""",2,0,0,4,121.0,76.0,123.8,5.28,"""PPO""","""Bronze""",1000,20,4,0,3.73,0.5714,6938.06,876.05,73.0,1,4672.59,4672.59,1,0,0,0,0,0,0,0,0,1,0,1,0,2,0,1,0,0
80185,79,"""Female""","""North""","""Urban""",12800.0,"""No HS""","""Married""","""Employed""",3,1,26.6,"""Never""","""Weekly""",2,0,0,3,131.0,79.0,97.3,4.82,"""POS""","""Gold""",1000,10,1,0,3.1,1.0,1632.61,445.1,37.09,4,297.27,1189.08,2,0,0,0,0,0,0,0,0,1,1,0,0,1,0,1,1,0
19865,68,"""Male""","""North""","""Rural""",40700.0,"""HS""","""Married""","""Retired""",5,3,31.5,"""Never""","""None""",1,0,0,4,160.0,84.0,129.5,5.51,"""HMO""","""Platinum""",500,20,10,0,3.9,1.0,7661.01,1538.02,128.17,0,0.0,0.0,3,1,0,0,0,0,1,0,0,0,1,1,0,2,1,0,1,0
76700,15,"""Male""","""North""","""Suburban""",15600.0,"""Some College""","""Married""","""Self-employed""",5,3,31.6,"""Never""","""None""",0,0,0,1,104.0,68.0,160.3,8.5,"""HMO""","""Silver""",500,20,5,0,3.89,0.2857,5130.27,820.63,68.39,0,0.0,0.0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0
92992,53,"""Male""","""Central""","""Suburban""",89600.0,"""Doctorate""","""Married""","""Self-employed""",2,0,30.5,"""Never""","""Daily""",3,0,0,2,136.0,83.0,171.0,5.2,"""POS""","""Platinum""",500,10,7,0,3.9,0.8681,1700.73,500.93,41.74,1,1002.24,1002.24,2,1,0,0,0,0,0,0,0,1,0,2,0,1,1,0,1,0


**List of features fields.**

In [4]:
primary_keys_list = ["person_id"]

features_list = [
    "age",
    "sex",
    "region",
    "urban_rural",
    "income",
    "education",
    "marital_status",
    "employment_status",
    "household_size",
    "dependents",
    "bmi",
    "smoker",
    "alcohol_freq",
    "visits_last_year",
    "hospitalizations_last_3yrs",
    "days_hospitalized_last_3yrs",
    "medication_count",
    "systolic_bp",
    "diastolic_bp",
    "ldl",
    "hba1c",
    "risk_score",
    "avg_claim_amount",
    "hypertension",
    "diabetes",
    "asthma",
    "copd",
    "cardiovascular_disease",
    "cancer_history",
    "kidney_disease",
    "liver_disease",
    "arthritis",
    "mental_health",
    "proc_imaging_count",
    "proc_surgery_count",
    "proc_physio_count",
    "proc_consult_count",
    "proc_lab_count",
    "had_major_procedure",
]

health_risk_factors_df = health_risk_factors_df.select(primary_keys_list + features_list)
print(f"Total # of features: {len(features_list)}")

Total # of features: 39


### 2. Apply data preprocessing

**Remove columns if they have high missingness rate.**

In [5]:
# Initialize parameters
tot_n_rows = health_risk_factors_df.shape[0]
missing_threshold = (
    0.4  # Columns w/ missing rate less than or equal to threshold are kept
)

# Calculate the percent of missing in each column
missing_count_df = health_risk_factors_df.null_count() / tot_n_rows
missing_count_df = missing_count_df.unpivot(
    on=features_list, variable_name="Variable Name", value_name="p_missing"
)

# Remove columns from data
columns_failed_threshold_list = (
    missing_count_df.filter(pl.col("p_missing") > missing_threshold)
    .select("Variable Name")
    .to_series()
)
preprocessed_risk_factors_df = health_risk_factors_df.drop(*columns_failed_threshold_list)
features_list = [c for c in features_list if c not in columns_failed_threshold_list]
print(
    f"Total # of columns removed due to high missing rate: {len(columns_failed_threshold_list)}"
)

Total # of columns removed due to high missing rate: 0


**Apply mean imputation to numeric fields**

In [6]:
# List of all numeric type columns
numeric_dtype_list = [col for col in preprocessed_risk_factors_df.select(cs.numeric()).columns if col not in primary_keys_list]

# Fill null with mean
preprocessed_risk_factors_df = preprocessed_risk_factors_df.with_columns(
    pl.col(numeric_dtype_list).fill_null(pl.col(numeric_dtype_list).mean())
)

preprocessed_risk_factors_df.select(primary_keys_list + numeric_dtype_list).show()

person_id,age,income,household_size,dependents,bmi,visits_last_year,hospitalizations_last_3yrs,days_hospitalized_last_3yrs,medication_count,systolic_bp,diastolic_bp,ldl,hba1c,risk_score,avg_claim_amount,hypertension,diabetes,asthma,copd,cardiovascular_disease,cancer_history,kidney_disease,liver_disease,arthritis,mental_health,proc_imaging_count,proc_surgery_count,proc_physio_count,proc_consult_count,proc_lab_count,had_major_procedure
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
75722,52.0,22700.0,3.0,1.0,27.4,2.0,0.0,0.0,4.0,121.0,76.0,123.8,5.28,0.5714,4672.59,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2.0,0.0,1.0,0.0
80185,79.0,12800.0,3.0,1.0,26.6,2.0,0.0,0.0,3.0,131.0,79.0,97.3,4.82,1.0,297.27,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
19865,68.0,40700.0,5.0,3.0,31.5,1.0,0.0,0.0,4.0,160.0,84.0,129.5,5.51,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,2.0,1.0,0.0,0.0
76700,15.0,15600.0,5.0,3.0,31.6,0.0,0.0,0.0,1.0,104.0,68.0,160.3,8.5,0.2857,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
92992,53.0,89600.0,2.0,0.0,30.5,3.0,0.0,0.0,2.0,136.0,83.0,171.0,5.2,0.8681,1002.24,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0,1.0,1.0,0.0,0.0


**Apply one-hot encoding to categorical fields**

In [7]:
# List of all string type columns
string_dtype_list = [
    name
    for name, dtype in preprocessed_risk_factors_df.select(features_list).schema.items()
    if dtype == pl.String
]

# Apply one-hot encoding
preprocessed_risk_factors_df = preprocessed_risk_factors_df.to_dummies(string_dtype_list)
ohe_vars_list = [col for col in preprocessed_risk_factors_df.columns for s in string_dtype_list if col.startswith(s)] 
preprocessed_risk_factors_df.select(primary_keys_list + ohe_vars_list).show()


person_id,sex_Female,sex_Male,sex_Other,region_Central,region_East,region_North,region_South,region_West,urban_rural_Rural,urban_rural_Suburban,urban_rural_Urban,education_Bachelors,education_Doctorate,education_HS,education_Masters,education_No HS,education_Some College,marital_status_Divorced,marital_status_Married,marital_status_Single,marital_status_Widowed,employment_status_Employed,employment_status_Retired,employment_status_Self-employed,employment_status_Unemployed,smoker_Current,smoker_Former,smoker_Never,alcohol_freq_Daily,alcohol_freq_None,alcohol_freq_Occasional,alcohol_freq_Weekly
i64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8
75722,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,1,0,0
80185,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,1
19865,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,1,0,0
76700,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,1,0,1,0,0
92992,0,1,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,1,0,0,0


**Convert to float data type**

In [8]:
preprocessed_risk_factors_df = preprocessed_risk_factors_df.select(pl.all().cast(pl.Float64))
preprocessed_risk_factors_df.show()

person_id,age,sex_Female,sex_Male,sex_Other,region_Central,region_East,region_North,region_South,region_West,urban_rural_Rural,urban_rural_Suburban,urban_rural_Urban,income,education_Bachelors,education_Doctorate,education_HS,education_Masters,education_No HS,education_Some College,marital_status_Divorced,marital_status_Married,marital_status_Single,marital_status_Widowed,employment_status_Employed,employment_status_Retired,employment_status_Self-employed,employment_status_Unemployed,household_size,dependents,bmi,smoker_Current,smoker_Former,smoker_Never,alcohol_freq_Daily,alcohol_freq_None,alcohol_freq_Occasional,alcohol_freq_Weekly,visits_last_year,hospitalizations_last_3yrs,days_hospitalized_last_3yrs,medication_count,systolic_bp,diastolic_bp,ldl,hba1c,risk_score,avg_claim_amount,hypertension,diabetes,asthma,copd,cardiovascular_disease,cancer_history,kidney_disease,liver_disease,arthritis,mental_health,proc_imaging_count,proc_surgery_count,proc_physio_count,proc_consult_count,proc_lab_count,had_major_procedure
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
75722.0,52.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,22700.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0,1.0,27.4,0.0,0.0,1.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0,4.0,121.0,76.0,123.8,5.28,0.5714,4672.59,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2.0,0.0,1.0,0.0
80185.0,79.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,12800.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,3.0,1.0,26.6,0.0,0.0,1.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0,3.0,131.0,79.0,97.3,4.82,1.0,297.27,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
19865.0,68.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,40700.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,5.0,3.0,31.5,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,4.0,160.0,84.0,129.5,5.51,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,2.0,1.0,0.0,0.0
76700.0,15.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,15600.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,5.0,3.0,31.6,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,104.0,68.0,160.3,8.5,0.2857,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
92992.0,53.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,89600.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0,30.5,0.0,0.0,1.0,1.0,0.0,0.0,0.0,3.0,0.0,0.0,2.0,136.0,83.0,171.0,5.2,0.8681,1002.24,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0,1.0,1.0,0.0,0.0


**Finalized list of features**

In [9]:
features_list = [col for col in preprocessed_risk_factors_df.columns if col not in primary_keys_list]
print(f"Total # of features: {len(features_list)}")
for col in features_list:
    print(col)

Total # of features: 63
age
sex_Female
sex_Male
sex_Other
region_Central
region_East
region_North
region_South
region_West
urban_rural_Rural
urban_rural_Suburban
urban_rural_Urban
income
education_Bachelors
education_Doctorate
education_HS
education_Masters
education_No HS
education_Some College
marital_status_Divorced
marital_status_Married
marital_status_Single
marital_status_Widowed
employment_status_Employed
employment_status_Retired
employment_status_Self-employed
employment_status_Unemployed
household_size
dependents
bmi
smoker_Current
smoker_Former
smoker_Never
alcohol_freq_Daily
alcohol_freq_None
alcohol_freq_Occasional
alcohol_freq_Weekly
visits_last_year
hospitalizations_last_3yrs
days_hospitalized_last_3yrs
medication_count
systolic_bp
diastolic_bp
ldl
hba1c
risk_score
avg_claim_amount
hypertension
diabetes
asthma
copd
cardiovascular_disease
cancer_history
kidney_disease
liver_disease
arthritis
mental_health
proc_imaging_count
proc_surgery_count
proc_physio_count
proc_consul

### 3. Perform topological matching using UMAP

**Apply dimensionality-reduction using UMAP**

We begin by performing dimensionality reduction on our data, keeping only the first 2 components. We apply UMAP in this example as our method of dimensionality reduction as it allows us to capture non-linearity in our lower dimensional space if needed.

In [10]:
# Number of neighbors each data point should have
n_neighbors = 50
# Number of components or dimensions to reduce the set of all features down to
n_components = 2
# Number of iterations or epochs to use for gradient descent algorithm
n_epochs = 50
# Learning rate for gradient descent
lr = 0.1

# Limit to only features fields
df = preprocessed_risk_factors_df.select(features_list)

# Run UMAP
mapper = UMAP(n_neighbors=n_neighbors, n_components=n_components, n_epochs=n_epochs, lr=lr)
embedding = mapper.fit_transform(df)

embedding.show()

umap_1,umap_2
f64,f64
3.440558,-12.13679
8.507531,10.069488
-21.086962,-14.505887
1.27633,-1.590244
-1.710164,-10.906935


**Bin and cluster data based on UMAP components**

We will now slice our UMAP component dimensions into multiple bins or intervals and perform clustering within each bin. It is important to note that the clustering is done using the raw, uncompressed, high-dimensional feature set rather than the dimensionally-reduced UMAP components; the UMAP components are only used to assign data points to bins. Because UMAP and, in general, any dimensionality reduction methods always introduces some amount of distortion or loss of information, it is better to cluster based on the raw features directly whenever possible. 

The bins in this situation mainly serve to guide local neighborhood division. We could, in theory, have skipped the binning and performed clustering immediately using the UMAP components, but standard clustering algorithms tend to create hard boundaries, assigning each data point to a single discrete cluster and as a consequence, possibly destroying any continuous topological structures like loops that may have existed in our data. In our binning process below, we actually allow each bin to overlap with one another so that a single data point can belong to multiple adjacent bins and be assigned to more than one cluster. This overlap connects local clusters into a simplical complex and maintains continuous topological shapes that traditional, hard clustering methods cannot.

In addition, global clustering algorithms often struggle when the data contains regions that vary drastically in density or structure from one another. By binning and localizing the data into smaller, bounded regions, we can better identify finer-grained sub-clusters within high-density data regions without missing coarse clusters in sparse data regions.


In [11]:
# Number of bins to slice each UMAP dimension into
# e.g. if n_components = 2 and n_bins = 5, then each dimension is sliced into 5 bin, giving a total of 5*5 = 25 grid spaces
n_bins = 4
# The amount of overlap between UMAP bins
overlap = 10.0
# The epsilon value to use for DBSCAN (density-based spatial clustering algorithm)
# epsilon represents the maximum distance between two samples for one to be considered in the neighborhood of the other
eps = 50.0
# The minimum number of data points within each bins in order to apply clustering
min_samples = 50


# Apply binning and clustering
clusters_df = mapper_bin_and_cluster(
    df_features=df,
    df_umap=embedding,
    n_bins=n_bins,
    overlap=overlap,
    eps=eps,
    min_samples=min_samples,
)

clusters_df.show()

Found 181 clusters in grid cell 1 / 16.
Found 181 clusters in grid cell 2 / 16.
Found 181 clusters in grid cell 3 / 16.
Found 181 clusters in grid cell 4 / 16.
Found 181 clusters in grid cell 5 / 16.
Found 181 clusters in grid cell 6 / 16.
Found 181 clusters in grid cell 7 / 16.
Found 181 clusters in grid cell 8 / 16.
Found 181 clusters in grid cell 9 / 16.
Found 181 clusters in grid cell 10 / 16.
Found 181 clusters in grid cell 11 / 16.
Found 181 clusters in grid cell 12 / 16.
Found 181 clusters in grid cell 13 / 16.
Found 181 clusters in grid cell 14 / 16.
Found 181 clusters in grid cell 15 / 16.
Found 181 clusters in grid cell 16 / 16.


bin_id,cluster_id,point_indices,node_size
str,i64,list[i64],i64
"""bin_0_0""",0,"[33, 1558, … 98179]",86
"""bin_0_0""",1,"[88, 2344, … 99765]",70
"""bin_0_0""",2,"[111, 172, … 99548]",78
"""bin_0_0""",3,"[125, 1339, … 99860]",71
"""bin_0_0""",4,"[166, 4127, … 99220]",67


**Assign to control and treatment group**

For each cluster, we randomly sample X numbers of members and assign to either a control or treatment group. Because we are sampling within each cluster, we ensure that we have a member with a similar matching set of characteristics between our control and treatment group. These two groups become the cohorts we monitor and experiment with for any A/B testing we would like to perform going forward.

In [12]:
# Number of points to sample for each bin_id and cluster_id combo
n_sample_per_bin_cluster = 5

# Expand the point_indices list into individual rows
exploded_df = (
    clusters_df
    .select("bin_id", "cluster_id", "point_indices")
    .explode("point_indices")
    .unique()
)

# Sample and extract unique point indices for control group
control_df = (
    exploded_df
    .group_by(["bin_id", "cluster_id"])
    .agg(pl.all().sample(n=n_sample_per_bin_cluster, seed=42))
    .explode("point_indices").sort(["bin_id", "cluster_id", "point_indices"])
)
control_group_indices = control_df.get_column("point_indices").to_list()
control_risk_factors_df = preprocessed_risk_factors_df[control_group_indices]

# Filter out members who have already been selected to be part of the control group
remaining_df = exploded_df.filter(~pl.col("point_indices").is_in(control_group_indices))

# Sample and extract unique point indices for treatment group from remaining unsampled members
treatment_df = (
    remaining_df
    .group_by(["bin_id", "cluster_id"])
    .agg(pl.all().sample(n=n_sample_per_bin_cluster, seed=42))
    .explode("point_indices").sort(["bin_id", "cluster_id", "point_indices"])
)
treatment_group_indices = treatment_df.get_column("point_indices").to_list()
treatment_risk_factors_df = preprocessed_risk_factors_df[treatment_group_indices]

print(f"Control group: {control_risk_factors_df.shape}")
print(f"Treatment group: {treatment_risk_factors_df.shape}")


Control group: (14480, 64)
Treatment group: (14480, 64)
